In [ ]:
import glob
import os
PROBES_DIR = os.path.join("Probe A (21.-22.06.2022)", "Probe A (21.-22.06.2022)")
print(f"PROBES_DIR: {PROBES_DIR}")
print(f"os.path.basename(PROBES_DIR): {os.path.basename(PROBES_DIR)}")


In [ ]:
glob.glob(os.path.join(PROBES_DIR, "sensor_4_*.csv"))[:5]

In [ ]:
import pandas as pd
import numpy as np

csv_path = os.path.join(PROBES_DIR, "sensor_4_sample_19.csv")
print(f"csv_path basename: {os.path.basename(csv_path)}")
print(f"csv_path dirname: {os.path.dirname(csv_path)}")
print(os.path.splitext(os.path.basename(csv_path)))
df = pd.read_csv(csv_path)

df.head()

In [ ]:
dfd = df.drop(columns=df.columns[0], inplace=False)
dfd.head()

In [ ]:
df.index.values

In [ ]:
wl_range = df.columns[1:].values
wl_range = np.array(wl_range, dtype=int)
wl_range

### Data Loading

#### Read and process csv

In [ ]:
def read_csv_data(path):
    df = pd.read_csv(path, encoding='ISO-8859-1')
    # Modification to the df
    df.drop(columns=df.columns[0], inplace=True)
    # average the intensity of the same wavelength
    wavelength = df.columns.to_numpy(dtype=int)
    intensity = df.mean(axis=0).to_numpy(dtype=float)
    return wavelength, intensity

# TODO: deal with empty cuvette, lamp on/off, etc.
def get_csv_paths(probes_dir, sensor, reference_sample: bool = False):
    if reference_sample:
        patterns = [
            os.path.join(probes_dir, f"{sensor}_lamp_on.csv"),
            os.path.join(probes_dir, f"{sensor}_lamp_off.csv"),
        ]
        paths = []
        for pattern in patterns:
            paths.extend(glob.glob(pattern))
        return paths
    else:
        pattern = os.path.join(probes_dir, f"{sensor}_sample_*.csv")
        return glob.glob(pattern)

In [ ]:
wavelength, intensity = read_csv_data(os.path.join(PROBES_DIR, "sensor_4_sample_19.csv"))
print(f"Wavelength length of sensor 4: {len(wavelength)}\n")
# print(f"Intensity length: {len(intensity)}")

In [ ]:
print(f"First 5 wavelength/intensity pairs:")
for i in range(min(5, len(wavelength))):
    print(f"  WL: {wavelength[i]}, Intensity: {intensity[i]:.2f}")

print(f"Last 5 wavelength/intensity pairs:")
for i in range(max(0, len(wavelength) - 5), len(wavelength)):
    print(f"  WL: {wavelength[i]}, Intensity: {intensity[i]:.2f}")

print(f"Wavelength range: {wavelength[0]} - {wavelength[-1]}")
print(f"Intensity range: {intensity[0]:.2f} - {intensity[-1]:.2f}")


#### Load and Combine all the data

In [ ]:
PROBE_LIST = ["Probe A", "Probe B", "Probe C", "Probe D"]
# Manual order, because the sensor_2 has the lowest wavelength
SENSOR_LIST = ["sensor_2", "sensor_1", "sensor_4", "sensor_3"]
get_reference_sample = True

all_sensor_data = {}
for sensor in SENSOR_LIST:
    # Initialize the sensor data list, to store the intensity data of each sample seperately
    sensor_data = []
    sample_paths = get_csv_paths(PROBES_DIR, sensor, get_reference_sample)
    for csv_path in sample_paths:
        _, intensity = read_csv_data(csv_path)
        sensor_data.append(intensity)

    # Combine the intensity data of all samples into a single array
    all_sensor_data[sensor] = np.vstack(sensor_data)
    print(f"Sensor Data Shape of {sensor}: {np.shape(all_sensor_data[sensor])} ({len(sample_paths)} samples)")


### Plotting

In [ ]:
import matplotlib.pyplot as plt
# Sample data
x = np.linspace(0, 10, 100)
y1 = 5*np.sin(x)
y2 = np.cos(x)
# Create figure and axes
# fig = plt.figure(figsize=(6, 4), dpi=150)
fig, axes = plt.subplots(2,2, dpi=150)
axes[0,0].plot(x, y1, label='sin(x)')
axes[0,0].plot(x, y2, label='cos(x)')
axes[0,0].legend()
axes[0,0].set_title('sin(x) and cos(x)')
axes[0,0].set_xlabel('x')
axes[0,0].set_ylabel('y')

In [ ]:
from typing import Any

font_size_title  = 10
font_size_ylabel = 8
font_size_xlabel = 8
font_size_ticks  = 6
font_size_legend = 6
grid_line_width  = 0.5
plot_line_width  = 0.5

fig, axes = plt.subplots(2, 2, figsize=(7.00,3.20), dpi=150)
axes = axes.ravel()

for ax, sensor in zip(axes, SENSOR_LIST):
    sample_paths = get_csv_paths(PROBES_DIR, sensor)
    for path in sample_paths:
        wavelength, intensity = read_csv_data(path)
        ax.plot(wavelength, intensity, linewidth=plot_line_width)
    ax.set_title(sensor)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Intensity")
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=font_size_ticks)
    ax.grid(True, linewidth=grid_line_width, ls='--')
fig.tight_layout()
plt.show()


### Statistics 

#### Maximum and Minimum Value
The intensity value of the sensor output should be further normalized based on the data of lamp on/off
